# StockVision AI — Notebook 01: Data Understanding

**Objective:** Understand the structure, quality and basic properties of the ingested market data.

**Analyst:** StockVision AI Pipeline  
**Data Source:** yfinance (NIFTY 50 Universe — 10 Indian stocks + benchmark)  
**Date Range:** 2019-01-01 to present

---

## Contents
1. Environment setup
2. Load data
3. Shape and schema
4. Data types and memory
5. Missing value analysis
6. Duplicate detection
7. Price distribution overview
8. Date continuity check
9. Data quality report
10. Key observations


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from src.utils.config import ALL_TICKERS, BENCHMARK_TICKER, COMPANY_INFO, settings
from src.database.queries import get_stock_prices
from src.processing.validate_data import generate_data_quality_report

# ── Styling ──────────────────────────────────────────────────────────────
plt.style.use('dark_background')
sns.set_palette('husl')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

ALL_SYMBOLS = ALL_TICKERS + [BENCHMARK_TICKER]
print(f'Configured stock universe: {len(ALL_SYMBOLS)} symbols')
print(ALL_SYMBOLS)

## 1. Load Data from PostgreSQL

In [ ]:
# Load all tickers
all_dfs = {}
for ticker in ALL_SYMBOLS:
    df = get_stock_prices(ticker, start_date=settings.historical_start_date)
    if not df.empty:
        df['trade_date'] = pd.to_datetime(df['trade_date'])
        df['ticker'] = ticker
        all_dfs[ticker] = df
    else:
        print(f'  ⚠️  No data for {ticker}')

print(f'\nLoaded data for {len(all_dfs)} tickers')

# Combine all
combined_df = pd.concat(all_dfs.values(), ignore_index=True)
print(f'Total rows: {len(combined_df):,}')

## 2. Schema and Shape

In [ ]:
print('=== Combined Dataset Shape ===')
print(f'Rows: {len(combined_df):,}  |  Columns: {combined_df.shape[1]}')
print()
print('=== Data Types ===')
print(combined_df.dtypes)
print()
print('=== Memory Usage ===')
mem_mb = combined_df.memory_usage(deep=True).sum() / 1024**2
print(f'Total memory: {mem_mb:.2f} MB')
print()
print('=== First 5 Rows (TCS.NS) ===')
combined_df[combined_df['ticker']=='TCS.NS'].head()

## 3. Rows per Ticker

In [ ]:
ticker_summary = combined_df.groupby('ticker').agg(
    rows=('trade_date', 'count'),
    date_start=('trade_date', 'min'),
    date_end=('trade_date', 'max'),
).reset_index()

ticker_summary['date_range_days'] = (ticker_summary['date_end'] - ticker_summary['date_start']).dt.days
ticker_summary['company'] = ticker_summary['ticker'].map(lambda t: COMPANY_INFO.get(t, {}).get('name', t))
ticker_summary['sector']  = ticker_summary['ticker'].map(lambda t: COMPANY_INFO.get(t, {}).get('sector', ''))

display(ticker_summary.sort_values('rows', ascending=False))

fig = px.bar(
    ticker_summary.sort_values('rows'),
    x='rows', y='ticker', orientation='h',
    color='sector', title='Trading Records per Ticker',
    template='plotly_dark', height=400
)
fig.show()

## 4. Missing Value Analysis

In [ ]:
null_counts = combined_df.isnull().sum()
null_pct    = (null_counts / len(combined_df) * 100).round(3)

missing_df = pd.DataFrame({'count': null_counts, 'pct': null_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

if missing_df.empty:
    print('✅ No missing values found — data quality is excellent!')
else:
    print('Missing Values Found:')
    display(missing_df)

    # Heatmap by ticker
    missing_by_ticker = combined_df.groupby('ticker').apply(lambda d: d.isnull().mean() * 100)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(missing_by_ticker, annot=True, fmt='.1f', cmap='Reds', ax=ax)
    ax.set_title('Missing Value % by Ticker and Column')
    plt.tight_layout()
    plt.show()

## 5. Duplicate Detection

In [ ]:
duplicates = combined_df.duplicated(subset=['ticker', 'trade_date'], keep=False)
n_dups = duplicates.sum()

if n_dups == 0:
    print('✅ No duplicate (ticker, trade_date) pairs found.')
else:
    print(f'⚠️ Found {n_dups} duplicate rows:')
    display(combined_df[duplicates].sort_values(['ticker', 'trade_date']))

## 6. Price Distribution Overview

In [ ]:
# Descriptive statistics by ticker
price_stats = combined_df.groupby('ticker')['close_price'].describe().round(2)
print('=== Close Price Statistics by Ticker ===')
display(price_stats)

# Distribution plots
fig = make_subplots(
    rows=3, cols=4,
    subplot_titles=[COMPANY_INFO.get(t, {}).get('name', t)[:20] for t in list(all_dfs.keys())[:11]]
)

for i, (ticker, df) in enumerate(list(all_dfs.items())[:11]):
    row = i // 4 + 1
    col = i % 4 + 1
    fig.add_trace(
        go.Histogram(x=df['close_price'], name=ticker, nbinsx=50,
                     marker_color=px.colors.qualitative.Set3[i % 12]),
        row=row, col=col
    )

fig.update_layout(
    template='plotly_dark', height=600,
    title='Close Price Distributions by Ticker',
    showlegend=False
)
fig.show()

## 7. OHLC Integrity Checks

In [ ]:
violations = {
    'high < low':           (combined_df['high_price'] < combined_df['low_price']).sum(),
    'close < 0':            (combined_df['close_price'] <= 0).sum(),
    'open < 0':             (combined_df['open_price'] <= 0).sum(),
    'volume < 0':           (combined_df['volume'] < 0).sum(),
    'high < close':         (combined_df['high_price'] < combined_df['close_price'] - 0.01).sum(),
    'low > close':          (combined_df['low_price']  > combined_df['close_price'] + 0.01).sum(),
}

viol_df = pd.DataFrame(list(violations.items()), columns=['Rule', 'Violations'])
print('=== OHLC Integrity Results ===')
display(viol_df)

total_viol = viol_df['Violations'].sum()
if total_viol == 0:
    print('\n✅ All OHLC integrity rules pass — data is clean!')
else:
    print(f'\n⚠️ {total_viol} total violations found — review cleaning pipeline.')

## 8. Date Continuity

In [ ]:
print('=== Date Coverage per Ticker ===')
date_coverage = []

for ticker, df in all_dfs.items():
    df_sorted = df.sort_values('trade_date')
    expected_business_days = pd.bdate_range(
        start=df_sorted['trade_date'].min(),
        end=df_sorted['trade_date'].max()
    )
    missing_days = max(0, len(expected_business_days) - len(df_sorted))
    coverage_pct = len(df_sorted) / len(expected_business_days) * 100

    date_coverage.append({
        'ticker':       ticker,
        'records':      len(df_sorted),
        'expected_bdays': len(expected_business_days),
        'missing_days': missing_days,
        'coverage_pct': round(coverage_pct, 1),
    })

cov_df = pd.DataFrame(date_coverage)
display(cov_df)

# Most missing days are Indian market holidays — expected
print('\n📌 Note: Missing business days are Indian market holidays (Holi, Diwali, etc.) — expected.')

## 9. Data Quality Report

In [ ]:
quality_report = generate_data_quality_report(all_dfs)
print('=== Automated Data Quality Report ===')
display(quality_report)

fig = px.bar(
    quality_report.sort_values('row_count'),
    x='row_count', y='ticker', orientation='h',
    title='Data Coverage by Ticker (rows)',
    template='plotly_dark',
    color='missing_days',
    color_continuous_scale='RdYlGn_r'
)
fig.show()

## 10. Key Observations

Document your findings here after running the notebook:

| # | Observation | Implication |
|---|---|---|
| 1 | Data loaded for `N` tickers covering `Y` years | Sufficient for modelling |
| 2 | Missing values in `adjusted_close` for some tickers | Forward-fill applied in cleaning |
| 3 | Zero OHLC violations | Ingestion pipeline is robust |
| 4 | Missing business days match Indian market calendar | Expected, not a data quality issue |
| 5 | Price scales differ significantly (RELIANCE ₹3000 vs ONGC ₹200) | Models trained on returns (%), not price levels |
